In [18]:
import mrpro

filename = "meas_MID00279_FID130349_COR_T1_TSE.h5"
path = f"/data/datasets/msk_mri_h5/h5/{filename}"

kdata = mrpro.data.KData.from_file(path,
    mrpro.data.traj_calculators.KTrajectoryCartesian(),
)
print(kdata)

KData on device "cpu" with (broadcasted) shape [48, 4, 1, 258, 640].
  data: Tensor<48, 4, 1, 258, 640>, |x| ∈ [4.66e-10, 0.00518], μ=4.16e-08+4.38e-09j, [-3.8347e-07-2.9979e-06j,  2.2943e-06+2.3306e-07j,  ...,  1.5881e-06+1.0126e-06j,  1.0177e-06+2.9602e-06j]
  traj: KTrajectory on device "cpu" with (broadcasted) shape [48, 1, 1, 258, 640].
     kz: Tensor<1, 1, 1, 1, 1>, constant 0
     ky: Tensor<48, 1, 1, 258, 1>, x ∈ [-258, 257], μ=-0.5, [-258., -256.,  ...,  255.,  257.]
     kx: Tensor<1, 1, 1, 1, 640>, x ∈ [-320, 319], μ=-0.5, [-320., -319.,  ...,  318.,  319.]
     grid_detection_tolerance: 0.001
     repeat_detection_tolerance: 0.001
  header:  KHeader on device "cpu" with (broadcasted) shape [48, 1, 1, 258, 1].
     recon_matrix: z=1, y=320, x=320
     encoding_matrix: z=1, y=646, x=640
     recon_fov: z=0.003, y=0.16, x=0.16
     encoding_fov: z=0.0045, y=0.322559998, x=0.32
     acq_info: AcqInfo<48, 1, 1, 258, 1>
     trajectory: KTrajectoryCartesian()
     lamor_frequenc

In [36]:
fft_op = mrpro.operators.FastFourierOp(
    dim=(-2, -1),
    recon_matrix=kdata.header.recon_matrix,
    encoding_matrix=kdata.header.encoding_matrix,
)
(img,) = fft_op.adjoint(kdata.data)

In [ ]:
import matplotlib.pyplot as plt
import torch

def show_images(*images: torch.Tensor, titles: list[str] | None = None) -> None:
    """Plot images."""
    n_images = len(images)
    _, axes = plt.subplots(1, n_images, squeeze=False, figsize=(n_images * 3, 3), dpi=200)
    for i in range(n_images):
        # axes[0][i].imshow(images[i], cmap='gray', vmin=0, vmax=images[i].max() * 0.6)
        axes[0][i].imshow(images[i], cmap='gray')
        axes[0][i].axis('off')
        if titles:
            axes[0][i].set_title(titles[i])
    plt.show()

    
magnitude_fully_sampled = img.abs().square().sum(dim=-4).sqrt().squeeze()


reconstruction = mrpro.algorithms.reconstruction.DirectReconstruction(kdata)
img = reconstruction(kdata)
print(img.shape)
# If there are multiple slices, ..., only the first one is selected
first_img = img.rss().squeeze()[0]  #  images, z, y, x
plt.imshow(first_img, cmap='gray')
plt.axis('off')
plt.show()

torch.Size([28, 1, 1, 1, 1])


IndexError: invalid index of a 0-dim tensor. Use `tensor.item()` in Python or `tensor.item<T>()` in C++ to convert a 0-dim tensor to a number

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mrpro.operators import FastFourierOp

print("kdata.data.shape:", kdata.data.shape)

# -------- 1) Detect number of averages from AcqInfo.idx --------
avg_idx = kdata.header.acq_info.idx.average  # same shape as data, but broadcasted
# flatten and get unique values (e.g. tensor([0., 1.]) for 2 averages)
avg_vals = torch.unique(avg_idx)
n_avg = int(avg_vals.numel())

print("unique average indices:", avg_vals.tolist())
print("n_avg:", n_avg)

# total “frames” in kdata.data (avg × slices)
data = kdata.data                      # (N_total, coils, 1, ky, kx)
n_total = data.shape[0]
assert n_total % n_avg == 0, "Total frames not divisible by n_avg"

n_slices = n_total // n_avg
print("inferred n_slices:", n_slices)

# -------- 2) Reshape to (n_avg, n_slices, ...) and average over avg dim --------
data_reshaped = data.reshape(n_avg, n_slices, *data.shape[1:])  # (n_avg, S, C, 1, ky, kx)
data_avg = data_reshaped.mean(dim=0)                            # (S, C, 1, ky, kx)
print("data_avg.shape:", data_avg.shape)

# -------- 3) Reconstruct from averaged k-space --------
fft_op = FastFourierOp(
    dim=(-2, -1),
    recon_matrix=kdata.header.recon_matrix,
    encoding_matrix=kdata.header.encoding_matrix,
)

(img_coils,) = fft_op.adjoint(data_avg)   # (S, C, Hy, Hx)

def rss_batch(coil_imgs: torch.Tensor) -> torch.Tensor:
    # coil_imgs: (S, C, H, W) -> (S, H, W)
    return coil_imgs.abs().square().sum(dim=1).sqrt()

rss = rss_batch(img_coils).detach().cpu().numpy()   # (S, H, W)
print("rss.shape:", rss.shape)

# -------- 4) Show one slice --------
sl = rss.shape[0] // 2
im = rss[sl]

vmin, vmax = np.percentile(im, [1, 99])
im_n = np.clip((im - vmin) / (vmax - vmin + 1e-8), 0, 1)

plt.figure(figsize=(5,5))
plt.imshow(im_n.squeeze(), cmap="gray")
plt.title(f"Averaged RSS slice {sl}")
plt.axis("off")
plt.show()
